# 03 — Full UCF101 pose extraction and sequence building (resumable)

This notebook expands the previous 5-class / 50-video pipeline to **all classes and all videos present in your `manifest.csv`**.

It processes **one video at a time**, saves sampled pose observations incrementally, skips completed videos on rerun, and builds the sequence arrays using disk-backed `.npy` memory maps instead of keeping the full dataset in RAM.

## Before running
- Keep the project folder and model/checkpoint paths the same as your current project.
- Run cells from top to bottom.
- Extraction can take a long time for the full dataset. You can interrupt the notebook; completed videos are retained and the next run resumes/skips them.
- This still uses YOLO + ViTPose pose extraction. It does not train the LSTM.
- The default is all manifest videos. For a pilot run, set `MAX_VIDEOS_PER_CLASS = 10` in the settings cell.
- The model uses the largest detected person independently in each sampled frame; multi-person identity tracking is not included.

In [1]:
from pathlib import Path
import sys
import re
import gc
import shutil
import time
import hashlib
import json
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
sys.path.insert(0, str(PROJECT_DIR))

from src.vitpose_model import (
    ViTPose,
    get_affine_transform,
    decode_heatmaps,
    draw_skeleton_on,
    IMG_H,
    IMG_W,
    IMG_MEAN,
    IMG_STD,
)

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_PATH = PROJECT_DIR / "notebooks" / "yolo11n.pt"
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"

assert CHECKPOINT_PATH.is_file(), f"ViTPose checkpoint not found: {CHECKPOINT_PATH}"
assert YOLO_PATH.is_file(), f"YOLO model file not found: {YOLO_PATH}"
assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"

# Keep CPU inference for this stable/resumable batch pilot.
DEVICE = "cpu"
torch.set_num_threads(2)

print("Loading ViTPose checkpoint on CPU...", flush=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = ViTPose()
model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()
SAVED_EPOCH = checkpoint.get("epoch", "unknown")
del checkpoint
gc.collect()
print("ViTPose loaded. Saved epoch:", SAVED_EPOCH, flush=True)

print("Loading YOLO detector...", flush=True)
detector = YOLO(str(YOLO_PATH))
print("YOLO loaded on CPU.", flush=True)

Loading ViTPose checkpoint on CPU...
ViTPose loaded. Saved epoch: 209
Loading YOLO detector...
YOLO loaded on CPU.


In [2]:
# ---------- User-adjustable settings ----------
# None means every class and every video in manifest.csv.
MAX_VIDEOS_PER_CLASS = None
RANDOM_SEED = 42

FRAME_STEP = 4
DETECTION_CONF = 0.25
DETECTION_CONF_FALLBACK = 0.10
YOLO_IMGSZ = 640
POSE_DRAW_THRESHOLD = 0.10
SAVE_PARTIAL_EVERY_SAMPLES = 5
SAVE_ONE_PREVIEW_PER_VIDEO = True

# Paths
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
BATCH_KEYPOINT_DIR = PROCESSED_DIR / "keypoints_batch_all"
PREVIEW_DIR = PROJECT_DIR / "outputs" / "pose_videos" / "batch_all_previews"
SELECTED_MANIFEST_PATH = PROCESSED_DIR / "selected_all_videos.csv"
EXTRACTION_REPORT_PATH = PROCESSED_DIR / "batch_all_extraction_report.csv"
SEQUENCE_DIR = PROCESSED_DIR / "sequences_all"

# Existing 5-class files can be reused, avoiding re-extraction for those videos.
OLD_KEYPOINT_DIR = PROCESSED_DIR / "keypoints_batch_5class"

for folder in [BATCH_KEYPOINT_DIR, PREVIEW_DIR, PROCESSED_DIR, SEQUENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)
required_cols = {"video_path", "label"}
missing_cols = required_cols - set(manifest.columns)
assert not missing_cols, f"Manifest is missing columns: {missing_cols}"
manifest["label"] = manifest["label"].astype(str)
manifest["video_path"] = manifest["video_path"].astype(str)
manifest = manifest.drop_duplicates(subset=["video_path"]).reset_index(drop=True)

print("Project:", PROJECT_DIR)
print("Manifest rows (unique videos):", len(manifest))
print("Classes in manifest:", manifest["label"].nunique())
print("Video counts per class (first 20):")
display(manifest.groupby("label").size().sort_values(ascending=False).head(20).rename("videos").reset_index())
print("Output keypoints:", BATCH_KEYPOINT_DIR)
print("Output sequences:", SEQUENCE_DIR)

Project: C:\Users\admin\ViTPose_Action_Recognition
Manifest rows (unique videos): 13320
Classes in manifest: 101
Video counts per class (first 20):


,label,videos
0,CricketShot,167
1,TennisSwing,166
2,PlayingDhol,164
3,HorseRiding,164
4,PlayingCello,164
5,BoxingPunchingBag,163
6,Drumming,161
7,ShavingBeard,161
8,Punch,160
9,BenchPress,160


Output keypoints: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_batch_all
Output sequences: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_all


In [3]:
def recording_group(video_path):
    stem = Path(str(video_path)).stem
    match = re.search(r"_g(\d+)_", stem, flags=re.IGNORECASE)
    return f"g{match.group(1)}" if match else stem

def choose_class_videos(class_df, n, seed):
    class_df = class_df.copy().sort_values("video_path").reset_index(drop=True)
    class_df["recording_group"] = class_df["video_path"].map(recording_group)
    representatives = class_df.drop_duplicates("recording_group", keep="first")
    rng = np.random.RandomState(seed)
    if n is None or n >= len(class_df):
        chosen = class_df
    elif len(representatives) >= n:
        chosen = representatives.sample(n=n, random_state=seed)
    else:
        remaining = class_df[~class_df["video_path"].isin(representatives["video_path"])]
        need = n - len(representatives)
        fill = remaining.sample(n=min(need, len(remaining)), random_state=seed)
        chosen = pd.concat([representatives, fill], ignore_index=True).head(n)
    return chosen.sort_values("video_path").reset_index(drop=True)

selected_parts = []
for class_offset, class_name in enumerate(sorted(manifest["label"].unique())):
    class_df = manifest[manifest["label"].eq(class_name)].copy()
    selected = choose_class_videos(class_df, MAX_VIDEOS_PER_CLASS, RANDOM_SEED + class_offset)
    selected_parts.append(selected)

selected_videos = pd.concat(selected_parts, ignore_index=True)
selected_videos["video_exists"] = selected_videos["video_path"].map(lambda p: Path(str(p)).is_file())
print("Selected videos per class:")
display(selected_videos.groupby("label").size().rename("selected_videos").reset_index())
print("Missing source files:", int((~selected_videos["video_exists"]).sum()))
if (~selected_videos["video_exists"]).any():
    display(selected_videos.loc[~selected_videos["video_exists"], ["label", "video_path"]].head(30))
selected_videos = selected_videos[selected_videos["video_exists"]].reset_index(drop=True)
assert len(selected_videos), "No source videos exist at the paths in manifest.csv."

# Copy already-extracted pilot outputs into the generalized folder, if present.
# The filename scheme is the same, so the extraction loop can skip them.
if OLD_KEYPOINT_DIR.is_dir():
    copied = 0
    for src in OLD_KEYPOINT_DIR.glob("*.npz"):
        if ".partial." in src.name or ".tmp." in src.name:
            continue
        dst = BATCH_KEYPOINT_DIR / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1
    print("Reused completed 5-class keypoint files:", copied)

selected_videos.to_csv(SELECTED_MANIFEST_PATH, index=False)
print("Saved selected manifest:", SELECTED_MANIFEST_PATH)
print("Videos queued:", len(selected_videos))

Selected videos per class:


,label,selected_videos
0,ApplyEyeMakeup,145
1,ApplyLipstick,114
2,Archery,145
3,BabyCrawling,132
4,BalanceBeam,108
...,...,...
96,VolleyballSpiking,116
97,WalkingWithDog,123
98,WallPushups,130
99,WritingOnBoard,152


Missing source files: 0
Reused completed 5-class keypoint files: 50
Saved selected manifest: C:\Users\admin\ViTPose_Action_Recognition\data\processed\selected_all_videos.csv
Videos queued: 13320


In [4]:
def pose_for_box_cpu(frame_rgb, bbox):
    """Estimate COCO-17 keypoints for one person's crop using the original model helpers."""
    x1, y1, x2, y2 = map(float, bbox)
    h, w = frame_rgb.shape[:2]
    x1 = float(np.clip(x1, 0, w - 1))
    x2 = float(np.clip(x2, 0, w))
    y1 = float(np.clip(y1, 0, h - 1))
    y2 = float(np.clip(y2, 0, h))
    bw, bh = x2 - x1, y2 - y1
    if bw < 2 or bh < 2:
        return None

    center = np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    transform = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(frame_rgb, transform, (IMG_W, IMG_H))
    crop = crop.astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float().unsqueeze(0)

    with torch.inference_mode():
        heatmaps = model(tensor)[0].float().cpu().numpy()
    keypoints = decode_heatmaps(heatmaps, center, scale).astype(np.float32)

    del tensor, heatmaps, crop
    if keypoints.shape != (17, 3):
        return None
    return keypoints


def safe_output_stem(label, video_path):
    safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", str(label)).strip("_") or "unknown"
    vp = Path(str(video_path))
    digest = hashlib.sha1(str(vp.resolve()).encode("utf-8")).hexdigest()[:10]
    return f"{safe_label}__{vp.stem}__{digest}"


def save_observations(npz_path, observations, label, video_path, fps, total_frames, frame_step):
    """Atomically save the currently completed sampled-frame observations."""
    if not observations:
        return
    arrays = {
        "keypoints": np.stack([o["keypoints"] for o in observations]).astype(np.float32),
        "boxes": np.stack([o["bbox"] for o in observations]).astype(np.float32),
        "frame_indices": np.asarray([o["frame_index"] for o in observations], dtype=np.int32),
        "detection_found": np.asarray([o["detection_found"] for o in observations], dtype=np.bool_),
        "has_person": np.asarray([o["pose_ok"] for o in observations], dtype=np.bool_),
        "confidence_used": np.asarray([o["confidence_used"] for o in observations], dtype=np.float32),
        "fallback_used": np.asarray([o["fallback_used"] for o in observations], dtype=np.bool_),
        "label": np.asarray(str(label)),
        "video_path": np.asarray(str(video_path)),
        "fps": np.asarray(float(fps), dtype=np.float32),
        "total_frames": np.asarray(int(total_frames), dtype=np.int32),
        "frame_step": np.asarray(int(frame_step), dtype=np.int32),
        "person_selection": np.asarray("largest_bbox_each_sampled_frame_no_tracking"),
    }
    tmp_path = npz_path.with_name(npz_path.stem + ".tmp.npz")
    np.savez_compressed(tmp_path, **arrays)
    tmp_path.replace(npz_path)


def load_observations(npz_path):
    """Load completed records from a partial NPZ to resume after interruption."""
    if not npz_path.is_file():
        return []
    with np.load(npz_path, allow_pickle=False) as d:
        keys = d["keypoints"].copy()
        boxes = d["boxes"].copy()
        indices = d["frame_indices"].copy()
        detected = d["detection_found"].copy()
        pose_ok = d["has_person"].copy()
        conf_used = d["confidence_used"].copy()
        fallback = d["fallback_used"].copy()
    return [
        {
            "keypoints": keys[i],
            "bbox": boxes[i],
            "frame_index": int(indices[i]),
            "detection_found": bool(detected[i]),
            "pose_ok": bool(pose_ok[i]),
            "confidence_used": float(conf_used[i]),
            "fallback_used": bool(fallback[i]),
        }
        for i in range(len(indices))
    ]


def build_report_row(video_path, label, status, observations, elapsed, output_path, error=""):
    total = len(observations)
    detections = sum(o["detection_found"] for o in observations)
    poses = sum(o["pose_ok"] for o in observations)
    fallback = sum(o["fallback_used"] for o in observations)
    return {
        "label": str(label),
        "video_path": str(video_path),
        "status": status,
        "sampled_frames": total,
        "detections_found": detections,
        "pose_successes": poses,
        "fallback_used_frames": fallback,
        "detection_rate": detections / total if total else 0.0,
        "pose_success_rate": poses / total if total else 0.0,
        "elapsed_seconds": round(float(elapsed), 1),
        "output_npz": str(output_path) if output_path else "",
        "error": error,
    }

print("Extraction helper functions defined.")

Extraction helper functions defined.


## Stage A — Batch-extract poses

Run this cell once. It loops over the selected videos automatically. It writes partial observations every few sampled frames and skips completed videos when rerun. The default processes all existing videos in the manifest.

In [ ]:
report_rows = []
run_started = time.time()

for video_position, (_, row) in enumerate(selected_videos.iterrows(), start=1):
    video_path = Path(str(row["video_path"]))
    label = str(row["label"])
    out_stem = safe_output_stem(label, video_path)
    final_npz = BATCH_KEYPOINT_DIR / f"{out_stem}.npz"
    partial_npz = BATCH_KEYPOINT_DIR / f"{out_stem}.partial.npz"
    preview_path = PREVIEW_DIR / f"{out_stem}_preview.jpg"

    print(f"\n[{video_position}/{len(selected_videos)}] {label}: {video_path.name}", flush=True)

    # Skip already-completed output files.
    if final_npz.is_file():
        try:
            loaded = load_observations(final_npz)
            report_row = build_report_row(video_path, label, "skipped_existing", loaded, 0, final_npz)
            print("  Existing complete NPZ found; skipping.", flush=True)
        except Exception as exc:
            print("  Existing NPZ could not be read; re-extracting:", repr(exc), flush=True)
            final_npz.unlink(missing_ok=True)
            report_row = None
        if report_row is not None:
            report_rows.append(report_row)
            pd.DataFrame(report_rows).to_csv(EXTRACTION_REPORT_PATH, index=False)
            continue

    observations = load_observations(partial_npz)
    already_done = {o["frame_index"] for o in observations}
    print("  Saved sampled frames to resume from:", len(observations), flush=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        report_row = build_report_row(video_path, label, "open_failed", observations, 0, partial_npz, "OpenCV could not open video")
        report_rows.append(report_row)
        pd.DataFrame(report_rows).to_csv(EXTRACTION_REPORT_PATH, index=False)
        cap.release()
        continue

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    total_frames_reported = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_index = 0
    new_samples_since_save = 0
    started = time.time()
    first_preview_saved = preview_path.is_file()
    video_error = ""

    try:
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break

            if frame_index % FRAME_STEP == 0 and frame_index not in already_done:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                det = detector.predict(
                    frame_bgr,
                    classes=[0],
                    conf=DETECTION_CONF,
                    imgsz=YOLO_IMGSZ,
                    device=DEVICE,
                    verbose=False,
                )[0]
                confidence_used = DETECTION_CONF
                fallback_used = False

                if det.boxes is None or len(det.boxes) == 0:
                    det = detector.predict(
                        frame_bgr,
                        classes=[0],
                        conf=DETECTION_CONF_FALLBACK,
                        imgsz=YOLO_IMGSZ,
                        device=DEVICE,
                        verbose=False,
                    )[0]
                    confidence_used = DETECTION_CONF_FALLBACK
                    fallback_used = True

                detection_found = det.boxes is not None and len(det.boxes) > 0
                bbox = np.full((4,), np.nan, dtype=np.float32)
                keypoints = np.full((17, 3), np.nan, dtype=np.float32)
                pose_ok = False

                if detection_found:
                    boxes = det.boxes.xyxy.cpu().numpy()
                    areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
                    bbox = boxes[int(np.argmax(areas))].astype(np.float32)
                    try:
                        estimated = pose_for_box_cpu(frame_rgb, bbox)
                        if estimated is not None and estimated.shape == (17, 3):
                            keypoints = estimated
                            pose_ok = bool(np.isfinite(keypoints[:, :2]).all(axis=1).any())
                    except Exception as exc:
                        print(f"  Pose error at frame {frame_index}: {exc!r}", flush=True)

                    if pose_ok and SAVE_ONE_PREVIEW_PER_VIDEO and not first_preview_saved:
                        preview_rgb = draw_skeleton_on(frame_rgb.copy(), keypoints, thresh=POSE_DRAW_THRESHOLD)
                        x1, y1, x2, y2 = map(int, bbox)
                        cv2.rectangle(preview_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        preview_path.parent.mkdir(parents=True, exist_ok=True)
                        cv2.imwrite(str(preview_path), cv2.cvtColor(preview_rgb, cv2.COLOR_RGB2BGR))
                        first_preview_saved = True

                observations.append({
                    "keypoints": keypoints.astype(np.float32),
                    "bbox": bbox.astype(np.float32),
                    "frame_index": int(frame_index),
                    "detection_found": bool(detection_found),
                    "pose_ok": bool(pose_ok),
                    "confidence_used": float(confidence_used),
                    "fallback_used": bool(fallback_used),
                })
                already_done.add(frame_index)
                new_samples_since_save += 1

                if len(observations) % 10 == 0:
                    print(
                        f"  samples={len(observations)} | detections={sum(o['detection_found'] for o in observations)} "
                        f"| pose successes={sum(o['pose_ok'] for o in observations)}",
                        flush=True,
                    )

                if new_samples_since_save >= SAVE_PARTIAL_EVERY_SAMPLES:
                    save_observations(partial_npz, observations, label, video_path, fps, total_frames_reported, FRAME_STEP)
                    new_samples_since_save = 0

                del frame_rgb, det
                if len(observations) % 10 == 0:
                    gc.collect()

            frame_index += 1

        # All original frames were read to EOF: extraction for this video is complete.
        save_observations(partial_npz, observations, label, video_path, fps, total_frames_reported, FRAME_STEP)
        if not observations:
            report_row = build_report_row(video_path, label, "no_samples", observations, time.time() - started, partial_npz, "No sampled frames extracted")
        else:
            partial_npz.replace(final_npz)
            report_row = build_report_row(video_path, label, "success", observations, time.time() - started, final_npz)

    except Exception as exc:
        video_error = repr(exc)
        # Preserve whatever was gathered before a normal Python exception.
        if observations:
            save_observations(partial_npz, observations, label, video_path, fps, total_frames_reported, FRAME_STEP)
        report_row = build_report_row(video_path, label, "error_partial_saved", observations, time.time() - started, partial_npz, video_error)
        print("  ERROR; partial file retained:", video_error, flush=True)

    finally:
        cap.release()
        gc.collect()

    report_rows.append(report_row)
    pd.DataFrame(report_rows).to_csv(EXTRACTION_REPORT_PATH, index=False)
    print(
        f"  Status={report_row['status']} | samples={report_row['sampled_frames']} | "
        f"poses={report_row['pose_successes']} | elapsed={report_row['elapsed_seconds']}s",
        flush=True,
    )

print("\nBATCH EXTRACTION CELL FINISHED", flush=True)
print("Total elapsed minutes:", round((time.time() - run_started) / 60, 1), flush=True)
print("Report:", EXTRACTION_REPORT_PATH, flush=True)
display(pd.DataFrame(report_rows))

In [ ]:
# Extraction audit
report = pd.read_csv(EXTRACTION_REPORT_PATH) if EXTRACTION_REPORT_PATH.is_file() else pd.DataFrame()
complete_files = [p for p in BATCH_KEYPOINT_DIR.glob("*.npz")
                  if ".partial." not in p.name and ".tmp." not in p.name]
partial_files = list(BATCH_KEYPOINT_DIR.glob("*.partial.npz"))
print("Completed NPZ files:", len(complete_files))
print("Partial NPZ files:", len(partial_files))
if not report.empty:
    print("\nStatus counts:")
    display(report["status"].value_counts().rename_axis("status").reset_index(name="videos"))
    print("\nPer-class report summary:")
    display(report.groupby("label").agg(
        videos=("video_path", "count"),
        pose_success_rate=("pose_success_rate", "mean"),
        sampled_frames=("sampled_frames", "sum")
    ).sort_values("videos", ascending=False).head(30))

---

## Stage B — Build fixed-length LSTM sequences

This stage reads completed per-video `.npz` files only. It splits by **source video before creating windows**, so windows from one video cannot leak across train/validation/test.

To avoid a large RAM spike, the builder makes two passes through one video at a time and writes the output arrays with NumPy memory maps. Sequence shape is `(N, 16, 51)` by default.

In [ ]:
SEQUENCE_LENGTH = 16
SEQUENCE_STRIDE = 4
MIN_VALID_FRAME_RATIO = 0.80
TRAIN_FRACTION = 0.60
VAL_FRACTION = 0.20
TEST_FRACTION = 0.20
SPLIT_SEED = 42

assert abs(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION - 1.0) < 1e-8
complete_npz_files = sorted(
    p for p in BATCH_KEYPOINT_DIR.glob("*.npz")
    if ".partial." not in p.name and ".tmp." not in p.name
)
print("Complete per-video NPZ files:", len(complete_npz_files))
assert complete_npz_files, "No complete per-video NPZ files. Run Stage A first."

In [ ]:
# First pass: read only each file's metadata and assign whole videos to splits.
video_meta = []
bad_files = []
for npz_path in complete_npz_files:
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            label = str(d["label"].item())
            video_path = str(d["video_path"].item())
            n_samples = int(len(d["frame_indices"]))
            n_pose_ok = int(d["has_person"].astype(bool).sum())
        video_meta.append({
            "npz_path": str(npz_path),
            "video_path": video_path,
            "label": label,
            "n_samples": n_samples,
            "n_pose_ok": n_pose_ok,
        })
    except Exception as exc:
        bad_files.append({"npz_path": str(npz_path), "error": repr(exc)})
        print("Skipping unreadable file:", npz_path.name, repr(exc))

video_meta_df = pd.DataFrame(video_meta).drop_duplicates(subset=["video_path"]).reset_index(drop=True)
print("Readable completed videos:", len(video_meta_df))
print("Classes represented:", video_meta_df["label"].nunique())
display(video_meta_df.groupby("label").size().sort_values(ascending=False).rename("videos").reset_index())

rng = np.random.RandomState(SPLIT_SEED)
split_rows = []
for label, group in video_meta_df.groupby("label", sort=True):
    paths = group["video_path"].to_numpy(dtype=object)
    paths = paths[rng.permutation(len(paths))]
    n = len(paths)
    if n < 3:
        print(f"WARNING: class {label!r} has only {n} videos; cannot guarantee all 3 splits.")
    n_test = max(1, int(round(n * TEST_FRACTION))) if n >= 3 else (1 if n >= 2 else 0)
    n_val = max(1, int(round(n * VAL_FRACTION))) if n >= 3 else (1 if n >= 3 else 0)
    if n_test + n_val >= n and n >= 3:
        n_test, n_val = 1, 1
    n_train = n - n_val - n_test
    for i, vp in enumerate(paths):
        split = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
        split_rows.append({"video_path": str(vp), "label": str(label), "split": split})

video_split_df = pd.DataFrame(split_rows)
video_split_df["video_id"] = np.arange(len(video_split_df), dtype=np.int32)
video_split = dict(zip(video_split_df["video_path"], video_split_df["split"]))
video_id_map = dict(zip(video_split_df["video_path"], video_split_df["video_id"]))
video_split_path = SEQUENCE_DIR / "video_split_manifest.csv"
video_split_df.to_csv(video_split_path, index=False)
print("Saved video split manifest:", video_split_path)
display(video_split_df.groupby(["label", "split"]).size().rename("videos").reset_index())

In [ ]:
def normalize_video_poses(keypoints, boxes, pose_ok):
    T = len(keypoints)
    normalized = np.zeros((T, 17, 3), dtype=np.float32)
    valid_frame = np.zeros(T, dtype=bool)
    for t in range(T):
        kp, box = keypoints[t], boxes[t]
        if not pose_ok[t] or not np.isfinite(box).all() or not np.isfinite(kp[:, :2]).all():
            continue
        x1, y1, x2, y2 = map(float, box)
        bw, bh = x2 - x1, y2 - y1
        if bw < 2 or bh < 2:
            continue
        normalized[t, :, 0] = (kp[:, 0] - x1) / bw
        normalized[t, :, 1] = (kp[:, 1] - y1) / bh
        scores = kp[:, 2].copy()
        scores[~np.isfinite(scores)] = 0.0
        normalized[t, :, 2] = np.clip(scores, 0.0, 1.0)
        normalized[t] = np.nan_to_num(normalized[t], nan=0.0, posinf=0.0, neginf=0.0)
        valid_frame[t] = True
    return normalized, valid_frame

def iter_windows(npz_path, label, video_path):
    with np.load(npz_path, allow_pickle=False) as d:
        kps = d["keypoints"].astype(np.float32)
        boxes = d["boxes"].astype(np.float32)
        frame_indices = d["frame_indices"].astype(np.int32)
        pose_ok = d["has_person"].astype(bool)
    if kps.ndim != 3 or kps.shape[1:] != (17, 3):
        return
    if not (len(kps) == len(boxes) == len(frame_indices) == len(pose_ok)):
        return
    normalized, valid_frame = normalize_video_poses(kps, boxes, pose_ok)
    T = len(normalized)
    if T < SEQUENCE_LENGTH:
        return
    starts = list(range(0, T - SEQUENCE_LENGTH + 1, SEQUENCE_STRIDE))
    final_start = T - SEQUENCE_LENGTH
    if starts and starts[-1] != final_start:
        starts.append(final_start)
    for start in starts:
        end = start + SEQUENCE_LENGTH
        valid_ratio = float(valid_frame[start:end].mean())
        if valid_ratio < MIN_VALID_FRAME_RATIO:
            continue
        window = np.nan_to_num(normalized[start:end], nan=0.0, posinf=0.0, neginf=0.0)
        yield window.reshape(SEQUENCE_LENGTH, 51), int(frame_indices[start]), int(frame_indices[end - 1]), valid_ratio

# Pass 1 over videos: count windows per split so disk-backed arrays can be sized exactly.
counts = {"train": 0, "val": 0, "test": 0}
per_video_counts = {}
for row in video_meta:
    split = video_split.get(row["video_path"])
    if split is None:
        continue
    n = sum(1 for _ in iter_windows(row["npz_path"], row["label"], row["video_path"]))
    per_video_counts[row["video_path"]] = n
    counts[split] += n

print("Sequence window counts:", counts)
print("Videos with at least one usable window:", sum(n > 0 for n in per_video_counts.values()))
trainable_labels = {
    row["label"] for row in video_meta
    if video_split.get(row["video_path"]) == "train" and per_video_counts.get(row["video_path"], 0) > 0
}
missing_train_labels = sorted(set(video_meta_df["label"]) - trainable_labels)
if missing_train_labels:
    raise ValueError(
        "These classes have no usable training sequences: " + ", ".join(missing_train_labels) +
        ". Inspect the extraction report and skeleton previews for these classes before training."
    )
assert counts["train"] > 0 and counts["val"] > 0 and counts["test"] > 0, (
    "At least one split has no valid sequences. Check extraction report or lower SEQUENCE_LENGTH / MIN_VALID_FRAME_RATIO."
)

In [ ]:
# Create memory-mapped arrays; this avoids holding the full sequence dataset in RAM.
label_names = sorted(video_meta_df["label"].unique().tolist())
label_to_id = {name: i for i, name in enumerate(label_names)}
label_map = {str(i): name for name, i in label_to_id.items()}
(SEQUENCE_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")

array_paths = {}
memmaps = {}
max_video_id = int(video_split_df["video_id"].max()) if len(video_split_df) else -1
for split in ["train", "val", "test"]:
    array_paths[split] = {
        "X": SEQUENCE_DIR / f"X_{split}.npy",
        "y": SEQUENCE_DIR / f"y_{split}.npy",
        "videos": SEQUENCE_DIR / f"videos_{split}.npy",
    }
    memmaps[split] = {
        "X": np.lib.format.open_memmap(
            array_paths[split]["X"], mode="w+", dtype=np.float32,
            shape=(counts[split], SEQUENCE_LENGTH, 51)
        ),
        "y": np.lib.format.open_memmap(
            array_paths[split]["y"], mode="w+", dtype=np.int64,
            shape=(counts[split],)
        ),
        # Store compact integer IDs instead of repeating long video paths for every window.
        "videos": np.lib.format.open_memmap(
            array_paths[split]["videos"], mode="w+", dtype=np.int32,
            shape=(counts[split],)
        ),
    }

offsets = {"train": 0, "val": 0, "test": 0}
video_sequence_summary = []
sequence_id = 0

for row in video_meta:
    vp, label, npz_path = row["video_path"], row["label"], row["npz_path"]
    split = video_split.get(vp)
    if split is None:
        continue
    before = offsets[split]
    for window, start_frame, end_frame, valid_ratio in iter_windows(npz_path, label, vp):
        idx = offsets[split]
        memmaps[split]["X"][idx] = window
        memmaps[split]["y"][idx] = label_to_id[label]
        memmaps[split]["videos"][idx] = video_id_map[vp]
        offsets[split] += 1
        sequence_id += 1
    video_sequence_summary.append({
        "video_id": int(video_id_map[vp]),
        "video_path": vp,
        "label": label,
        "split": split,
        "sequence_windows": int(offsets[split] - before),
        "sampled_frames": int(row["n_samples"]),
        "pose_successes": int(row["n_pose_ok"]),
        "source_npz": npz_path,
    })

for split in ["train", "val", "test"]:
    for arr in memmaps[split].values():
        arr.flush()
    assert offsets[split] == counts[split], (
        f"Count mismatch for {split}: wrote {offsets[split]}, expected {counts[split]}"
    )
    print(
        f"Saved {split}: X shape={memmaps[split]['X'].shape}, "
        f"windows={offsets[split]}, source={array_paths[split]['X']}"
    )

del memmaps
gc.collect()

sequence_manifest = pd.DataFrame(video_sequence_summary)
sequence_manifest_path = SEQUENCE_DIR / "video_sequence_summary.csv"
sequence_manifest.to_csv(sequence_manifest_path, index=False)
video_split_df.to_csv(SEQUENCE_DIR / "video_split_manifest.csv", index=False)

# Verify video-level split separation.
video_sets = {
    split: set(video_split_df.loc[video_split_df["split"].eq(split), "video_path"].astype(str))
    for split in ["train", "val", "test"]
}
assert not (video_sets["train"] & video_sets["val"])
assert not (video_sets["train"] & video_sets["test"])
assert not (video_sets["val"] & video_sets["test"])

print("\nSaved label map:", label_map)
print("Per-video sequence summary:", sequence_manifest_path)
print("Sequence data folder:", SEQUENCE_DIR.resolve())
print("No source-video overlap between splits.")

In [ ]:
# Final verification
print("DATASET SUMMARY")
print("Sequence length:", SEQUENCE_LENGTH)
print("Features per frame: 51 (17 joints x/y/confidence)")
print("Number of classes in label map:", len(label_map))
print("Total completed source videos:", len(video_meta_df))
for split in ["train", "val", "test"]:
    X = np.load(SEQUENCE_DIR / f"X_{split}.npy", mmap_mode="r", allow_pickle=False)
    y = np.load(SEQUENCE_DIR / f"y_{split}.npy", mmap_mode="r", allow_pickle=False)
    vids = np.load(SEQUENCE_DIR / f"videos_{split}.npy", mmap_mode="r", allow_pickle=False)
    print(f"{split}: X={X.shape}, y={y.shape}, unique videos={len(np.unique(vids))}")
    print("  windows by class:", {
        label_map[str(int(k))]: int(v)
        for k, v in zip(*np.unique(y, return_counts=True))
    })
display(video_split_df.groupby(["label", "split"]).size().rename("videos").reset_index())
print("\\nPer-video sequence summary saved at:", SEQUENCE_DIR / "video_sequence_summary.csv")
print("\nImportant: if any class has no train windows, inspect that class's extraction report before training.")

## Output folders

- `data/processed/keypoints_batch_all/` — one `.npz` per processed video, plus resumable partial files if interrupted.
- `data/processed/batch_all_extraction_report.csv` — extraction success and pose rates.
- `data/processed/sequences_all/` — memory-mapped `X_*.npy` arrays, labels, source-video IDs, split manifest and sequence manifest.
- `outputs/pose_videos/batch_all_previews/` — one skeleton preview per video when possible.

For the full dataset, extraction can take many hours. First confirm that the manifest contains all intended UCF101 classes and valid file paths. Do not start the full extraction if the manifest points to only a subset.